# K-Nearest Neighbours (KNN) — Heart Disease Classification

Implements KNN classification on the heart disease dataset, with manual k-tuning, `GridSearchCV` cross-validation, and a `Pipeline` combining scaling and classification.

**How KNN works:**  
For each test sample, the algorithm finds the k training points closest to it (by Euclidean distance) and predicts the majority class among those neighbours. Because it relies on distance, features must be on a comparable scale — StandardScaler is essential here.

**Key concepts covered:**
- Manual k sweep (k = 3, 5, 7, 9) to observe the bias–variance trade-off
- `GridSearchCV` with 5-fold cross-validation and recall scoring to find the optimal k systematically
- `Pipeline` — chains scaler and classifier into a single estimator so scaling is always applied consistently and data leakage between folds is prevented

**Best result:** k = 7 achieves ~91.8% accuracy and 90.6% recall on the test set.

In [1]:
import pandas as pd
from sklearn.metrics import precision_score, accuracy_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
heart_df = pd.read_csv("heart.csv")

X = heart_df.drop("target", axis=1)
y = heart_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# KNN classifies by computing distances between points, so features on different scales
# (e.g. age in years vs cholesterol in mg/dL) would unfairly dominate the distance calculation.
# StandardScaler brings all features to mean=0, std=1 before any distance is measured.
# fit_transform on training data, transform-only on test data to prevent data leakage.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [3]:
knn_classifier = KNeighborsClassifier(n_neighbors=3)
knn_classifier.fit(X_train_scaled, y_train)

y_pred = knn_classifier.predict(X_test_scaled)

print("recall score: ", recall_score(y_test, y_pred))
print("accuracy score: ", accuracy_score(y_test, y_pred))
print("precision score: ", precision_score(y_test, y_pred))

recall score:  0.78125
accuracy score:  0.8524590163934426
precision score:  0.9259259259259259


In [4]:
# K=5
knn_classifier = KNeighborsClassifier(n_neighbors=5)
knn_classifier.fit(X_train_scaled, y_train)

y_pred = knn_classifier.predict(X_test_scaled)

print("recall score: ", recall_score(y_test, y_pred))
print("accuracy score: ", accuracy_score(y_test, y_pred))
print("precision score: ", precision_score(y_test, y_pred))

recall score:  0.875
accuracy score:  0.9016393442622951
precision score:  0.9333333333333333


In [5]:
# K=7 - Best score till now
knn_classifier = KNeighborsClassifier(n_neighbors=7)
knn_classifier.fit(X_train_scaled, y_train)

y_pred = knn_classifier.predict(X_test_scaled)

print("recall score: ", recall_score(y_test, y_pred))
print("accuracy score: ", accuracy_score(y_test, y_pred))
print("precision score: ", precision_score(y_test, y_pred))

recall score:  0.90625
accuracy score:  0.9180327868852459
precision score:  0.9354838709677419


In [6]:
# K=9
knn_classifier = KNeighborsClassifier(n_neighbors=9)
knn_classifier.fit(X_train_scaled, y_train)

y_pred = knn_classifier.predict(X_test_scaled)

print("recall score: ", recall_score(y_test, y_pred))
print("accuracy score: ", accuracy_score(y_test, y_pred))
print("precision score: ", precision_score(y_test, y_pred))

recall score:  0.875
accuracy score:  0.9016393442622951
precision score:  0.9333333333333333


In [ ]:
# GridSearchCV — exhaustive search over the parameter grid using cross-validation.
# scoring="recall" means the best k is chosen to maximise recall across folds,
# which is the right choice for a medical dataset where missing a true positive is costly.
from sklearn.model_selection import GridSearchCV

classifier = KNeighborsClassifier()
param_grid = {"n_neighbors": [3, 5, 7, 9]}

classifierCV = GridSearchCV(
    classifier,
    param_grid,
    cv=5,            # 5-fold cross-validation
    scoring="recall" # optimise for recall, not accuracy
)

classifierCV.fit(X_train_scaled, y_train)

y_pred = classifierCV.predict(X_test_scaled)

print("recall score: ", recall_score(y_test, y_pred))
print("accuracy score: ", accuracy_score(y_test, y_pred))
print("precision score: ", precision_score(y_test, y_pred))

res = pd.DataFrame(classifierCV.cv_results_)
print(res[["param_n_neighbors", "mean_test_score"]])  # CV recall per k value

print(classifierCV.best_params_)  # k chosen by cross-validation

In [ ]:
# Pipeline — wraps the scaler and classifier into a single estimator.
# Benefits over manual scaling:
# 1. Prevents data leakage: the scaler is re-fit on each training fold during CV,
#    never seeing validation data.
# 2. Cleaner code: fit/predict calls handle both steps automatically.
# 3. Easier deployment: the whole pipeline can be serialised and served as one object.
#
# Note: GridSearchCV parameter names use the format "stepname__paramname"
# so "knn__n_neighbors" refers to the n_neighbors parameter of the "knn" step.
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

param_grid = {"knn__n_neighbors": [3, 5, 7, 9]}

classifierCV = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="recall"
)

classifierCV.fit(X_train, y_train)

y_pred = classifierCV.predict(X_test)

print("recall score: ", recall_score(y_test, y_pred))
print("accuracy score: ", accuracy_score(y_test, y_pred))
print("precision score: ", precision_score(y_test, y_pred))

print(classifierCV.best_params_)